# Machine Learning - Álgebra Linear

**Tarefa:**  Não se aplica<br>
**Dataset:** Não se aplica  
**Objetivo:** Entender intuitivamente a finalidade da Álgebra Linear dentro de Aprendizado de Máquina <br>
**Última Alteração**: 25/05/2026

## 1. Conceitos

### Vetores

Vetores são pontos e direções. Um vetor é apenas uma lista de números e estes números signnificam algo, pois são coordenadas no espaço.

Na IA, vetores representam tudo:
* Uma palavra -> Um vetor de 500 números (seu "significado" no espaço de imersão)
* Uma imagem -> Um vetor de milhões de pixels
* Um usuário -> Um vetor de preferências

Um vetor 2d [3,2]:

| X | Y | Ponto |
|---|---|-------|
| 3 | 2 |(3, 2) |


O que o produto escalar de dois vetores mede? <br>
R: O quão similar ou alinhados os dois vetores são. Resultado positivo significa mesma direção (similar), resultado igual a 0 significa perpendicular (não relacionados) e resultado negativo significa direção oposta. Tal conceito é a base da busca por similaridade em IA.

Dados vetores a e b:
* MESMA DIREÇÃO: a . b > 0 (similar)
* PERPENDICULAR: a . b = 0 (sem relação)
* DIREÇÃO OPOSTA: a . b < 0 (dispar) 



### Matrizes

Matrizes são transformações. Uma matriz transforma um vetor em outro:; pode rotacionar, escalar, alargar ou projetá-lo.

Em IA, matrizes SÃO modelos:
* Pesos de redes neurais: matrizes que transformam entradas em saídas
* Pontos de atenção: matrizes que decidem em que focar
* Embeddings: matrizes que mapeiam palavras para vetores

### Independência Linear

Os vetores serão linearmente independentes se nenhum vetor do conjunto puder ser escrito como uma combinação dos outros. Se vetores a, b e c forem independentes, eles abrangirão um espaço tridimensional, mas se um deles for uma combinação dos outros, eles abrangirão apenas um plano.

**Qual a importância disso?**

Sua matriz de características deve possuir colunas linearmente independentes. Se duas características forem perfeitamente correlacionadas (linearmente dependentes), o modelo não conseguirá distinguir seus efeitos. Isso causa multicolinearidade na regressão. A matriz de pesos se torna instável e pequenas alterações na entrada produzirão **GRANDES** oscilações na saída.

## 2. Implementações

OBS.: Num cenário real isso não é praticável, pois o trabalho será feito com **Numpy**, as implementações abaixo são apenas para compreensão de conceitos e seus aspectos "por baixo dos panos".

### Vetor

In [ ]:
# Implementação básica de um vetor

class Vetor:
    def __init__(self, componentes):
        self.componentes = list(componentes)
        self.dim = len(self.componentes)

    def __add__(self, outro):
        return Vetor([a + b for a, b in zip(self.componentes, outro.componentes)])

    def __sub__(self, outro):
        return Vetor([a - b for a, b in zip(self.componentes, outro.componentes)])

    def produto(self, outro):
        return sum(a * b for a, b in zip(self.componentes, outro.componentes))

    def magnitude(self):
        return sum(x**2 for x in self.componentes) ** 0.5

    def normaliza(self):
        mag = self.magnitude()
        return Vetor([x / mag for x in self.componentes])

    def similaridade_cosseno(self, outro):
        return self.produto(outro) / (self.magnitude() * outro.magnitude())

    def __repr__(self):
        return f"Vetor({self.componentes})"

a = Vetor([1, 2, 3])
b = Vetor([4, 5, 6])

print(f"a + b = {a + b}")
print(f"a · b = {a.produto(b)}")
print(f"|a| = {a.magnitude():.4f}")
print(f"Similaridade de Cosseno = {a.similaridade_cosseno(b):.4f}")



a + b = Vetor([5, 7, 9])
a · b = 32
|a| = 3.7417
Similaridade de Cosseno = 0.9746


### Matrizes

In [ ]:
# Implementação básica de uma matriz

class Matriz:
    def __init__(self, linhas):
        # recebe uma lista de listas (as linhas da matriz) e calcula o shape
        self.linhas = [list(linha) for linha in linhas]
        self.shape = (len(self.linhas), len(self.linhas[0]))

    def __matmul__(self, outro):
        if isinstance(outro, Vetor):
            return Vetor([
                sum(self.linhas[i][j] * outro.componentes[j] for j in range(self.shape[1]))
                for i in range(self.shape[0])
            ])
        linhas = []
        for i in range(self.shape[0]): #linha da 1ª matriz
            linha = []
            for j in range(outro.shape[1]): # coluna da 2ª matriz
                linha.append(sum(
                    self.linhas[i][k] * outro.linhas[k][j]
                    for k in range(self.shape[1])
                ))
            linhas.append(linha)
        return Matriz(linhas)

    def transposta(self):
        return Matriz([
            [self.linhas[j][i] for j in range(self.shape[0])]
            for i in range(self.shape[1])
        ])

    def __repr__(self):
        return f"Matriz({self.linhas})"

rotacao_90 = Matriz([[0, -1], [1, 0]])
ponto = Vetor([3, 1])

rotacionada = rotacao_90.__matmul__(ponto)
print(f"Original: {ponto}")
print(f"Rotacionada: {rotacionada}")



Original: Vetor([3, 1])
Rotacionada: Vetor([-1, 3])


### A importância disso pra IA

In [3]:
import random

random.seed(25)
pesos = Matriz([[random.gauss(0, 0.1) for _ in range(3)] for _ in range(2)])
vetor_entrada = Vetor([1.0, 0.5, -0.3])

saida = pesos @ vetor_entrada # @ é o mesmo que usar __matmul__
print(f"Entrada (3D): {vetor_entrada}")
print(f"Saída (2D): {saida}")
print(f"Praticamente é o que uma camada de rede neural realiza, uma multiplicação de mmatrizes.")

Entrada (3D): Vetor([1.0, 0.5, -0.3])
Saída (2D): Vetor([-0.09536460736672236, 0.022258128553292535])
Praticamente é o que uma camada de rede neural realiza, uma multiplicação de mmatrizes.


### Na prática com o Numpy

In [4]:
import numpy as np
a = np.array([1, 2, 3], dtype=float)
b = np.array([4, 5, 6], dtype=float)

print(f"a + b = {a + b}")
print(f" a . b = {np.dot(a, b)}")
print(f" |a| = {np.linalg.norm(a):.4f}")
print(f"cosseno = {np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)):.4f}")

W = np.random.randn(2, 3) * 0.1
x = np.array([1.0, 0.5, -0.3])
print(f"Wx = {W @ x}")

a + b = [5. 7. 9.]
 a . b = 32.0
 |a| = 3.7417
cosseno = 0.9746
Wx = [-0.06238179  0.0222714 ]
